## 6.3 安全通信仿真

在上一节中，我们建立了双节点 FER/BER 的基线。本节引入 ECDH 配对和 AES-CCM 加密，对比明文与密文在相同信道条件下的 FER，验证安全通道的建立流程和加密对链路性能的影响。

本节学习大纲如下：

- 安全通信流程演示：配对 + 加密传输
- 明文 vs 密文 FER 对比扫描
- 加密开销分析

### 本实验涉及的关键文件

```
src/nearlink_sdr/
├── node.py                  <- SleNode: start_pairing / enable_encryption
├── mac/
│   ├── security_manager.py   <- PairingManager: ECDH 密钥协商
│   │                            FrameCryptoContext: AES-CCM 加解密
│   └── link_manager.py       <- 链路状态机 (含 PAIRING 状态)
├── phy/
│   └── channel.py            <- ChannelModel: AWGN 加噪
└── sim/
    └── link_sim.py           <- sim_dual_node_secure_link: 安全通信仿真
                                   sim_encrypted_vs_plain: 明文/密文对比
```

---

### 1. 安全通信原理

星闪 SLE 标准（7.2.8）定义了基于 ECDH 的配对与密钥协商流程，随后使用 AES-CCM 对数据帧进行加密和完整性保护。

#### ECDH 密钥协商

1. **G 节点发起配对**：发送配对请求，携带本端 ECDH 公钥
2. **T 节点响应**：收到请求后生成自己的 ECDH 密钥对，回复公钥
3. **双方计算共享密钥**：利用对端公钥 + 本端私钥 → 相同的 session_key
4. **配对完成**：双方标记 `paired=True`，session_key 用于后续加密

#### AES-CCM 加密

使用配对得到的 session_key 创建 `FrameCryptoContext`：

- **加密**：AES-128-CTR 模式加密载荷 + 帧头关联数据（AAD）
- **完整性**：CCM 模式生成 4 字节 MIC（Message Integrity Code），接收端验证
- **IV 管理**：由 IV base + 帧计数器组成，每帧递增防止重放攻击

#### 加密开销

加密后的帧比明文帧多出 MIC 字段（4 字节），相当于有效载荷增加了少量开销。在 FER-SNR 曲线上，加密与明文曲线几乎重合——因为加密不改变编码调制方式，仅改变载荷的比特内容。

---
### 2. 安全通信流程演示


上述流程中涉及的三个安全函数：

**`enable_encryption=True`** — NodeConfig 参数。启用后，`transmit()` 在 `mac_to_iq` 之前自动调用 `_crypto.encrypt()` 加密载荷，`receive()` 在 `iq_to_mac` 之后自动调用 `_crypto.decrypt()` 解密并校验 MIC。

---

**`start_pairing(peer_address)`** — 发起 ECDH 配对流程。生成本端 ECDH 密钥对（公钥 + 私钥），返回待发送给对端的配对信令列表。

```python
def start_pairing(self, peer_address: bytes) -> list:
    is_g = self._link_mgr.role == Role.G_NODE    # 根据本端角色决定 PairingManager 模式
    self._pairing = PairingManager(                # 创建配对管理器
        is_g_node=is_g,                            #   G 节点发起配对挑战
        local_address=self.config.address,         #   本端地址
        peer_address=peer_address)                 #   对端地址
    return self._pairing.start()                   # 生成配对信令列表 (含 ECDH 公钥)
```

其中 `PairingManager`（位于 `security_manager.py`）内部管理完整的 ECDH 握手状态机：
- `_pairing.start()` — 生成初始配对消息（G 节点发送公钥挑战，T 节点等待）
- `_pairing.process_message(msg)` — 处理对端消息并推进状态：`INIT → CHALLENGE_SENT → RESPONSE_SENT → COMPLETED`
- `_pairing.state` — 当前配对状态，`PairingState.COMPLETED` 表示双方已计算出共享 `session_key`

---

**`process_pairing_message(msg)`** — 处理对端发来的配对信令，返回需要发给对端的响应信令。双方交替调用直到配对完成。

```python
def process_pairing_message(self, msg: object) -> list:
    if self._pairing is None:                     # 未发起配对则忽略
        return []
    responses = self._pairing.process_message(msg) # 处理信令, 推进状态机
    if self._pairing.state == PairingState.COMPLETED:  # 配对完成!
        self._setup_crypto()                       # 用 session_key 创建 FrameCryptoContext
        self._link_mgr.process_event(              # 通知链路管理器: 配对完成
            Event(EventType.PAIRING_COMPLETE))
    return responses
```

其中 `_setup_crypto()` 利用协商出的 `session_key` 创建 `FrameCryptoContext` 对象（AES-128-CTR + CCM MIC），赋值给 `self._crypto`。此后 `transmit()` 和 `receive()` 内部自动调用 `self._crypto.encrypt()` / `decrypt()`。

---

配对成功后，`g.stats["paired"]` 和 `t.stats["paired"]` 均为 True，后续所有数据帧自动加密传输。

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.mac.link_manager import Role
from nearlink_sdr.node import NodeConfig, NodeRole, SleNode

g_addr = b"\x01\x02\x03\x04\x05\x06"
t_addr = b"\x0A\x0B\x0C\x0D\x0E\x0F"

# ---- 创建加密节点 ----
g = SleNode(config=NodeConfig(
    address=g_addr, role=NodeRole.G_NODE,
    frame_type=2, mcs_index=7,
    enable_encryption=True))     # 启用加密
t = SleNode(config=NodeConfig(
    address=t_addr, role=NodeRole.T_NODE,
    frame_type=2, mcs_index=7,
    enable_encryption=True))

# ---- 建链 ----
g.start_advertising()
t.start_scanning()
t.connect(g_addr)
g.accept_connection(t_addr, Role.G_NODE)
print(f"1. Link: G={g.state.name}, T={t.state.name}")

# ---- 配对 (ECDH 密钥协商) ----
g_msgs = g.start_pairing(t_addr)         # G 发起配对
for msg in g_msgs:
    t_msgs = t.process_pairing_message(msg)  # T 处理配对消息
for msg in t_msgs:
    g_msgs = g.process_pairing_message(msg)  # G 处理 T 的回复
paired = g.stats["paired"] and t.stats["paired"]
print(f"2. Pairing: {'OK' if paired else 'FAIL'}")
print(f"   Shared key established: {paired}")

# ---- 加密数据交换 ----
rng = np.random.default_rng(42)
payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
g.send(payload)
tx = g.transmit()                         # encrypt() 在 transmit() 内部调用
from nearlink_sdr.sim.link_sim import _channel_impair
rx_iq = _channel_impair(tx.iq, 10.0, "awgn",
    6.0, 0.0, "none", g._tx_config.sps, rng)
from nearlink_sdr.mac.frame import AsyncDataFrame
frame = AsyncDataFrame(segment_type=0, data=payload)
rx = t.receive(rx_iq, len(frame.pack()))  # decrypt() 在 receive() 内部调用
print(f"3. Encrypted data: {'OK' if rx.success else 'FAIL'}")
g.disconnect()
print(f"4. Disconnect: G={g.state.name}")

---

### 3. 明文 vs 密文 FER 对比

在以上代码的基础上进行 SNR 扫描封装成 sim_encrypted_vs_plain() 函数，可以执行以下代码来查看：


In [ ]:
!cat -n src/nearlink_sdr/sim/link_sim.py | sed -n "2532,2651p"

调用 `sim_encrypted_vs_plain()` 来测量安全链路配对成功率和在同一 SNR 范围下对比明文和加密链路的 FER：

In [ ]:
from nearlink_sdr.sim.link_sim import sim_encrypted_vs_plain

snr_range = np.arange(0, 16, 1) 
evp = sim_encrypted_vs_plain(
    snr_range_db=snr_range,
    n_frames=50, mcs_index=7, seed=42)

print(f"{'SNR':>5s}  {'Plain':>10s}  {'Encrypted':>10s}")
print("-" * 30)
for s, fp, fe in zip(snr_range, evp["fer_plain"], evp["fer_encrypted"]):
    print(f"{s:5.0f}  {fp:10.4f}  {fe:10.4f}")

# 配对成功率
from nearlink_sdr.sim.link_sim import sim_secure_link
pair_ok = sum(1 for i in range(100)
              if sim_secure_link(snr_range_db=np.array([10.0]), n_frames=1,
                                 mcs_index=7, seed=42 + i)["pairing_ok"])
print(f"\nPairing success rate @ SNR=10 dB: {pair_ok}/100")


---

### 4. 密文和明文的 FER 对比曲线

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(evp["snr_db"], [max(f, 1e-4) for f in evp["fer_plain"]],
            "o-", lw=2, label="Plain")
ax.semilogy(evp["snr_db"], [max(f, 1e-4) for f in evp["fer_encrypted"]],
            "s--", lw=2, label="Encrypted")
ax.set_xlabel("SNR (dB)"); ax.set_ylabel("FER")
ax.set_title("Encrypted vs Plain FER (MCS=7)")
ax.legend(); ax.grid(True, which="both", ls="--", alpha=0.5)
ax.set_ylim(bottom=1e-4); plt.show()

---

### 5. 实验分析

- **配对流程**：观察 ECDH 密钥协商是否在 SNR=10 dB 下达到 100% 成功率
- **密文 vs 明文 FER**：两线是否基本重合？如果出现偏差，可能的原因是什么

## 课后实践

请补全下方 ECDH 配对消息交换中的 **3 处空缺**（每处一行代码），完成 G/T 双节点的安全配对握手。

要求：

1. 补全 G 节点发起配对
2. 补全 T 节点处理配对消息（G → T）
3. 补全 G 节点处理配对消息（T → G）

完成后运行 ，观察配对是否成功。

In [ ]:
%%writefile pairing_practice.py
import sys
sys.path.insert(0, "../src")
from nearlink_sdr.mac.link_manager import Role
from nearlink_sdr.node import NodeConfig, NodeRole, SleNode

g_addr = b"\x01\x02\x03\x04\x05\x06"
t_addr = b"\x0A\x0B\x0C\x0D\x0E\x0F"

g = SleNode(config=NodeConfig(
    address=g_addr, role=NodeRole.G_NODE,
    frame_type=2, mcs_index=7, enable_encryption=True))
t = SleNode(config=NodeConfig(
    address=t_addr, role=NodeRole.T_NODE,
    frame_type=2, mcs_index=7, enable_encryption=True))

# ---- 建链 ----
g.start_advertising(); t.start_scanning()
t.connect(g_addr); g.accept_connection(t_addr, Role.G_NODE)
print(f"Link: G={g.state.name}, T={t.state.name}")

# ==== 补全配对消息交换（3处空缺）====
# G 发起配对, 发送公钥挑战
g_msgs = ______________  # 1: G 发起配对 （补全）

# T 收到 G 的消息, 处理并生成响应
for msg in g_msgs:
    t_msgs = ______________  # 2: T 处理配对消息 （补全）

# G 收到 T 的响应, 完成协商
for msg in t_msgs:
    g_msgs = ______________  # 3: G 处理配对消息 （补全）

# 检查配对结果
paired = g.stats["paired"] and t.stats["paired"]
print(f"Pairing: {"OK" if paired else "FAIL"}")
print(f"Shared key established: {paired}")


执行以下命令进行编译并验证结果：


In [ ]:
!python pairing_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/06.03_answer.txt
